# 01a_Metadaten_Intake

Dieses Notebook übernimmt **nur** den Daten-Intake und eine **minimale Harmonisierung** der Metadaten aus zwei Quellen:
- `Top_100/top_metadata.json`
- `Normal_100/normal_metadata.json`

Die Idee ist, den Intake **sauber von der Qualitätssicherung** (01b) und dem **Feature-Engineering** (02a) zu trennen.  
So wird der Prozess **reproduzierbar** und **prüfbar**.

## Governance & Guardrails

- **Quellenfixierung:** Es werden ausschließlich die genannten JSON-Dateien gelesen (keine Ordnerscans, keine Überraschungen).
- **Deterministische Verarbeitung:** Es gibt keine Zufallsauswahl (z. B. beim Deduplizieren nehmen wir stets den “neuesten” Datensatz pro `video_id`).
- **Zeitnormalisierung:** Zeitspalten werden, sofern vorhanden, in **UTC** geparst. Einheitliche Zeitzonen sind wichtig für spätere Zeit-Features.
- **Provenienzpflicht:** Für jede `video_id` wird festgehalten, aus welcher Quelle sie stammt (`source_file`, `group`) – wichtig für Transparenz, Debugging und Bericht.


## 1) Setup & Pfade

**Ziel:** Pfade so setzen, dass das Notebook sowohl im Repo als auch in einer isolierten Umgebung läuft.  
Wir definieren:
- `PROJECT_ROOT`: Wurzel des Repos  
- `TIKTOK_DIR`: Pfad zu den JSON-Quellen  
- **Fallbacks** für den Fall, dass die Dateien in dieser Umgebung unter `/mnt/data/...` liegen  
Wir geben aus, welche Datei tatsächlich verwendet wird (Sichtprüfung).


In [13]:

from pathlib import Path
import pandas as pd, numpy as np, json, re, warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..").resolve()
TIKTOK_DIR = PROJECT_ROOT / "tiktok_100_vs_100"

TOP_JSON = TIKTOK_DIR / "Top_100" / "top_metadata.json"
NORMAL_JSON = TIKTOK_DIR / "Normal_100" / "normal_metadata.json"

FALLBACK_TOP = Path("/mnt/data/top_metadata.json")
FALLBACK_NORMAL = Path("/mnt/data/normal_metadata.json")
if not TOP_JSON.exists() and FALLBACK_TOP.exists():
    TOP_JSON = FALLBACK_TOP
if not NORMAL_JSON.exists() and FALLBACK_NORMAL.exists():
    NORMAL_JSON = FALLBACK_NORMAL

print("TOP_JSON:", TOP_JSON, TOP_JSON.exists())
print("NORMAL_JSON:", NORMAL_JSON, NORMAL_JSON.exists())


TOP_JSON: C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemester 5\F5 PALG\Viralitaetsanalyse\tiktok_100_vs_100\Top_100\top_metadata.json True
NORMAL_JSON: C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemester 5\F5 PALG\Viralitaetsanalyse\tiktok_100_vs_100\Normal_100\normal_metadata.json True


## 2) Helper (Loader & minimale Harmonisierung)

**`read_json_table`**
- liest JSON-Dateien robust (Liste von Objekten oder `{ "data": [...] }`)  
- normalisiert die Spaltennamen (`snake_case`)  
- ergänzt **Provenienz**:`source_file` und **Gruppe**:`group` (`top` oder `normal`)

**`unify_types`**
- parst Zeitspalten in **UTC** (`upload_time`, `create_time`, `timestamp`, `create_date`)
- castet Zahlenfelder vorsichtig in numerische Typen
- sorgt für **kanonische Namen** (z. B. `like_count` → `likes`, `id` → `video_id`)
- setzt `video_id` als **String** (wichtig für ID-Stabilität und CSV/Parquet-Kompatibilität)

**`pick_one_row_per_video`**
- bei mehrfachen Zeilen je `video_id` wird **deterministisch** die neueste Zeile gewählt  
  (Sortierung nach Zeitspalte, dann `drop_duplicates`)


In [14]:

def snake(s: str): 
    import re
    return re.sub(r"\W+", "_", str(s).strip()).strip("_").lower()

def cols_to_snake(df):
    df = df.copy()
    df.columns = [snake(c) for c in df.columns]
    return df

def read_json_table(path: Path, group_name: str):
    import json, pandas as pd
    assert path.exists(), f"Datei fehlt: {path}"
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
            df = pd.DataFrame(data["data"])
        else:
            df = pd.DataFrame(data if isinstance(data, list) else [data])
    except Exception:
        df = pd.read_json(path, lines=True)
    df = cols_to_snake(df)
    df["source_file"] = str(path)
    df["group"] = group_name
    return df

def unify_types(df):
    df = df.copy()
    for c in ["upload_time","create_time","timestamp","create_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], utc=True, errors="coerce")
    for c in ["duration_s","duration","creator_follower_count","creator_posts_count",
              "views","view_count","likes","like_count","comments","comment_count","shares","share_count","rank"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in ["creator_verified","verified"]:
        if c in df.columns:
            df[c] = df[c].fillna(False).astype(bool)
    rename = {
        "verified":"creator_verified",
        "like_count":"likes", "view_count":"views", "comment_count":"comments", "share_count":"shares",
        "duration":"duration_s",
        "id":"video_id", "aweme_id":"video_id", "tiktok_id":"video_id"
    }
    for k,v in rename.items():
        if k in df.columns and v not in df.columns:
            df = df.rename(columns={k:v})
    if "video_id" in df.columns:
        df["video_id"] = df["video_id"].astype(str)
    return df

def pick_one_row_per_video(df):
    time_col = next((c for c in ["upload_time","create_time","timestamp","create_date"] if c in df.columns), None)
    if time_col:
        df = df.sort_values(["video_id", time_col], ascending=[True, False])
    else:
        df = df.sort_values(["video_id"])
    return df.drop_duplicates("video_id", keep="first").reset_index(drop=True)


## 3) Quellen laden

**Warum wir testen, dass es genau 2 Dateien sind:**  
Damit wir sicherstellen, dass wir wirklich die geplanten Datensätze verarbeiten – nicht mehr, nicht weniger.  
So ist der Intake **stabil** und **prüfbar**.


In [15]:

sources = []
if TOP_JSON.exists():
    sources.append(("top", TOP_JSON))
if NORMAL_JSON.exists():
    sources.append(("normal", NORMAL_JSON))

print("Gefundene Dateien:", len(sources), [str(p) for _, p in sources])
assert len(sources) == 2, "Es sollten genau zwei JSON-Dateien gefunden werden."

parts = []
for grp, path in sources:
    dfp = read_json_table(path, grp)
    dfp = unify_types(dfp)
    parts.append(dfp)

meta_raw = pd.concat(parts, ignore_index=True, sort=False)
meta_raw.shape, meta_raw.head(3)


Gefundene Dateien: 2 ['C:\\Users\\lremm\\OneDrive\\Desktop\\FHdW\\Fachsemester 5\\F5 PALG\\Viralitaetsanalyse\\tiktok_100_vs_100\\Top_100\\top_metadata.json', 'C:\\Users\\lremm\\OneDrive\\Desktop\\FHdW\\Fachsemester 5\\F5 PALG\\Viralitaetsanalyse\\tiktok_100_vs_100\\Normal_100\\normal_metadata.json']


((200, 12),
    rank group             video_id             uploader    likes  comments  \
 0     1   top  7461297586738154798  6751051329931609094  8500000     22800   
 1     2   top  7544520710744558903              7445731  7000000     10100   
 2     3   top  7538167343642397973  6596805238354558982  4300000     22600   
 
       views  shares  engagement_score                title  duration_s  \
 0  79400000  504800           8545600     Goodbye TikTok 🫡          12   
 1  60900000  624300           7020200       I can’t stop 😩          10   
 2  49600000   58100           4345200  en los comentarios            6   
 
                                          source_file  
 0  C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemes...  
 1  C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemes...  
 2  C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemes...  )

## 4) Harmonisierung & eine Zeile pro `video_id`

**Warum “eine Zeile pro `video_id`”?**  
Alle späteren Features (Audio, Video, Metadaten) werden auf **Video-Ebene** zusammengeführt.  
Doppelte Zeilen würden hier zu fehlerhaften Merges führen.


In [16]:

assert "video_id" in meta_raw.columns, "Spalte 'video_id' fehlt in den JSONs."
before = meta_raw.shape[0]
meta_one = pick_one_row_per_video(meta_raw)
after = meta_one.shape[0]
print(f"Dedupliziert: {before} → {after}")
meta_one.head(5)


Dedupliziert: 200 → 200


,rank,group,video_id,uploader,likes,comments,views,shares,engagement_score,title,duration_s,source_file
0,10,top,7219026508239686954,6751051329931609094,2000000,7103,18000000,31700,2014206,Hype House is going through some changes. Than...,8,C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemes...
1,19,top,7231352152743152942,6751051329931609094,1500000,1492,15300000,18500,1502984,I somehow got unbreakable hangers,28,C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemes...
2,89,normal,7236485370542624042,6751051329931609094,87600,213,4000000,340,88026,Rate our mini house #MakingMyWay,18,C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemes...
3,22,normal,7239370205254929710,6751051329931609094,47900,222,3700000,181,48344,#ad Are you team chipIN or team chipOUT? @lays...,38,C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemes...
4,44,normal,7303636279936322862,6751051329931609094,59100,214,8300000,155,59528,How would you do gift wrapping without seeing?...,61,C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemes...


## 5) (Optional) Proxy-Label für QC

Wir erzeugen **`is_viral_proxy`** (Top=1, Normal=0), **nur** um in der EDA Muster zu erkennen.  
> **Wichtig:** Das ist **kein** finales Trainingslabel.  
Sobald eure **echte Label-Definition** steht (z. B. “> X Views in Y Tagen”), gehört die Datei `data/labels.csv` ins Repo und wir mergen diese **in 01b**.


In [17]:

proxy_map = {"top":1, "normal":0}
if "group" in meta_one.columns and "is_viral_proxy" not in meta_one.columns:
    meta_one["is_viral_proxy"] = meta_one["group"].map(proxy_map).fillna(0).astype(int)
meta_one[["video_id","group","is_viral_proxy"]].head(5)


,video_id,group,is_viral_proxy
0,7219026508239686954,top,1
1,7231352152743152942,top,1
2,7236485370542624042,normal,0
3,7239370205254929710,normal,0
4,7303636279936322862,normal,0


## 6) Artefakte speichern

**Warum wir überhaupt speichern:**  
- **`metadata_raw`** ist die stabile, reprozierbare Übergabe an 01b (QC/Cleaning).  
- **`metadata_provenance`** dokumentiert die Herkunft – unverzichtbar für Nachvollziehbarkeit, Debugging und weitere Bearbeitung.

**Format-Strategie:**  
- Versuch **Parquet** (effizient), aber **Fallback auf CSV**, wenn die Engine fehlt.  
- So kann das Notebook **auf jeder Maschine** laufen, ohne dass ihr zuerst `pyarrow` installieren müsst.

> 📌 *Platzhalter – Team-Entscheidung:*  
> „Welches **Austauschformat** wollen wir im Repo standardisieren (Parquet vs. CSV)?“  
> **Vorschlag:** Parquet als Default, CSV nur zusätzlich wenn nötig.


In [18]:
from pathlib import Path

def save_parquet_or_csv(df, base_name: str):
    """
    Versucht, eine DataFrame als Parquet zu speichern.
    Wenn pyarrow/fastparquet fehlt, speichert automatisch als CSV.
    Gibt den tatsächlich geschriebenen Pfad zurück.
    """
    base = Path(base_name)
    parquet_path = base.with_suffix(".parquet")
    csv_path = base.with_suffix(".csv")
    try:
        # Versuch: Parquet
        df.to_parquet(parquet_path, index=False)
        print(f"Gespeichert (Parquet): {parquet_path.resolve()}")
        return parquet_path
    except Exception as e:
        print("ℹ️ Parquet-Export nicht verfügbar (oder fehlgeschlagen). Fallback → CSV.")
        print("   Grund:", repr(e))
        df.to_csv(csv_path, index=False)
        print(f"Gespeichert (CSV): {csv_path.resolve()}")
        return csv_path

# 1) Roh-harmonisierte Tabelle
raw_written = save_parquet_or_csv(meta_one, "metadata_raw")

# 2) Provenienz
prov = meta_one[["video_id","source_file","group"]].drop_duplicates("video_id")
prov_path = Path("metadata_provenance.csv")
prov.to_csv(prov_path, index=False)
print("Provenienz gespeichert:", prov_path.resolve())


ℹ️ Parquet-Export nicht verfügbar (oder fehlgeschlagen). Fallback → CSV.
   Grund: ImportError("Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.\nA suitable version of pyarrow or fastparquet is required for parquet support.\nTrying to import the above resulted in these errors:\n - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.\n - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.")
Gespeichert (CSV): C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemester 5\F5 PALG\Viralitaetsanalyse\Metadaten_Analyse\metadata_raw.csv
Provenienz gespeichert: C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemester 5\F5 PALG\Viralitaetsanalyse\Metadaten_Analyse\metadata_provenance.csv


### Übergabe an 01b_Metadaten_Cleaning_QC

- In **01b** prüfen wir **Schema, Missingness, Leakage** und schreiben das abgesicherte `metadata_base.(parquet/csv)`.  
- Ab da beginnt **02a** mit deterministischem Feature-Engineering.

